# Group 5 VIA project: setting and testing an accuracy target

**Question:** how accurate does our screening model need to be, and how close is a first model on the Jhpiego + IARC data?

**What this notebook does**
1. States an accuracy target from the literature (below).
2. Loads all 300 labelled images: IARC VIA Image Bank (185 cases with a post-acetic-acid photo) and Jhpiego flashcards (115 cards; photo cropped from the scan, label read from the answer card).
3. Runs four experiments with 5-fold cross-validation: a frozen-feature baseline (same idea as the proposal feasibility test), fine-tuned EfficientNet-B0, fine-tuned MobileNetV3 (phone-sized), and an average of the two fine-tuned models.
4. Reports AUC and sensitivity at fixed specificity with 95% bootstrap confidence intervals, and checks the target.

Runtime: GPU (T4). About 20 minutes end to end. Data comes from the shared Drive folder, or from `via_data.zip` if it has been uploaded to `/content` through the Files panel.

*Built with help from an AI coding assistant (Cursor); results were run and checked in this notebook.*

**Team copy:** preliminary experiments from Kavin (first run Nov 2 2026, Colab T4). Results are summarised in the Findings section at the end; rerunning reads data from the team folder and writes results to `AI ML Final Project/results/accuracy_target`.

## Literature review: what accuracy to aim for

| Source | Setting | Reference standard | Result |
|---|---|---|---|
| Sauvaget et al., *Int J Gynecol Obstet* 2011 (26 studies) | Human VIA, asymptomatic women | Histology, CIN2+ | Sensitivity **80%**, specificity **92%** (sensitivity ranged 41–92% across studies) |
| Qiao et al., *J Obstet Gynaecol Res* 2015 (29 studies, bivariate model) | Human VIA | Histology, CIN2+ | Sensitivity **73.2%**, specificity **86.7%** |
| Hu et al., *JNCI* 2019 (NCI "automated visual evaluation") | Deep learning (Faster R-CNN) on 60,000 cervigrams, 9,406 women | Histology, CIN2+ | AUC **0.91** (0.89–0.93) vs **0.69** for expert reading of the same images |
| Indonesia smartphone VIA app, *BMC Res Notes* 2022 | 199 women, smartphone photos, 65 test images | Oncologist VIA reading | AUC **0.85** (0.66–1.0), sensitivity 80%, specificity 96.4% |
| Systematic review, *Diagnostics* 2023 (11 AI-on-VIA studies) | Mostly small, selected image sets | Mixed | Sensitivity 0.22–0.93, specificity 0.67–0.95 |
| Prior class team (same project) | Jhpiego + IARC | Expert VIA | ~0.80 three-class accuracy |

**Our target (screening question: does this woman need referral, i.e. VIA-positive or suspicious of cancer?)**
- **Minimum bar, "as good as a typical VIA provider":** sensitivity ≥ 73% at specificity ≥ 87%.
- **Stretch goal, "as good as pooled VIA / published smartphone AI":** sensitivity ≥ 80% at specificity ≥ 92%, and AUC ≥ 0.85.

Overall accuracy is reported for comparison with the prior team, but the target is set on sensitivity at a fixed specificity, because missing a precancer costs far more than an extra referral.

**Caveat:** our labels are expert VIA readings, not biopsy results (only about half the IARC cases have histology). So this measures how well the model reproduces an expert VIA reader, while the literature numbers measure detection of CIN2+ on histology. The last section uses the IARC cases that have histology for a small like-for-like check.

In [ ]:
import os, sys, math, random, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
UPLOADED_ZIP = '/content/via_data.zip'  # optional: data zip uploaded via the Files panel instead of Drive
if IN_COLAB and os.path.exists(UPLOADED_ZIP):
    import zipfile
    zipfile.ZipFile(UPLOADED_ZIP).extractall('/content')
    DATA_ROOT = '/content/via_data'
    OUT = '/content/results/accuracy_target'
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/AI ML Final Project'
    candidates = [f'{PROJECT}/data from original team', f'{PROJECT}/data/jhpiego_iarc', '/content/drive/MyDrive/cervical cancer data']
    DATA_ROOT = next(p for p in candidates if os.path.isdir(f'{p}/IARCImageBankVIA'))
    OUT = f'{PROJECT}/results/accuracy_target'
else:
    DATA_ROOT = os.environ['VIA_DATA_ROOT']
    OUT = os.environ.get('VIA_OUT', 'results/accuracy_target')
os.makedirs(OUT, exist_ok=True)
print('data:', DATA_ROOT)
print('outputs:', OUT)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, confusion_matrix

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
NUM_WORKERS = 2 if device.type == 'cuda' else 0
print('device:', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

# Experiment settings
N_FOLDS = 5
IMG = 320
BATCH = 16
EPOCHS = int(os.environ.get('VIA_EPOCHS', 25))
LR = 1e-3

# Target (from the literature table above)
MIN_SPEC, MIN_SENS = 0.867, 0.73
STRETCH_SPEC, STRETCH_SENS, STRETCH_AUC = 0.92, 0.80, 0.85
REFERENCE_POINTS = {  # name -> (sensitivity, specificity)
    'Human VIA, pooled (Sauvaget 2011)': (0.80, 0.92),
    'Human VIA, bivariate (Qiao 2015)': (0.732, 0.867),
    'Smartphone AI (Indonesia 2022)': (0.80, 0.964),
}

## Data

In [ ]:
# Data loading helpers (same as final_project/via_data.py in the repo)
"""Load the Jhpiego + IARC VIA images into one labelled table.

Labels follow the expert VIA reading: Negative / Positive / Suspicious of cancer.
`refer` is the screening target: 1 if Positive or Suspicious of cancer.
"""
import os

import numpy as np
import pandas as pd
from PIL import Image

CLASSES = ["Negative", "Positive", "Suspicious of cancer"]


def _longest_run(mask):
    best, start, best_span = 0, None, (0, len(mask))
    for i, v in enumerate(list(mask) + [False]):
        if v and start is None:
            start = i
        elif not v and start is not None:
            if i - start > best:
                best, best_span = i - start, (start, i)
            start = None
    return best_span


def crop_flashcard(img, inset=0.02):
    """Cut the cervix photo out of a scanned Jhpiego flashcard page.

    The photo sits in the top half of the page; the logo and text are excluded
    by searching only there and keeping the largest block of non-white rows.
    """
    img = img.convert("RGB")
    a = np.asarray(img).astype(np.int16)
    top = a[: a.shape[0] // 2]
    spread = top.max(axis=2) - top.min(axis=2)
    ink = (spread > 30) | (top.mean(axis=2) < 170)
    r0, r1 = _longest_run(ink.mean(axis=1) > 0.25)
    c0, c1 = _longest_run(ink[r0:r1].mean(axis=0) > 0.5)
    dy, dx = int((r1 - r0) * inset), int((c1 - c0) * inset)
    return img.crop((c0 + dx, r0 + dy, c1 - dx, r1 - dy))


def load_table(root, jhpiego_labels_csv):
    """root: folder containing IARCImageBankVIA/ and via/."""
    iarc = os.path.join(root, "IARCImageBankVIA")
    meta = pd.read_excel(os.path.join(iarc, "Cases Meta data.xlsx"))
    imgs = pd.read_excel(os.path.join(iarc, "Cases - Images.xlsx"))
    after = imgs[imgs["Type"] == "After application of acetic acid"]
    after = after.drop_duplicates("CaseNumber")
    case_dirs = {}
    for d in os.listdir(iarc):
        p = os.path.join(iarc, d)
        if os.path.isdir(p):
            for f in os.listdir(p):
                case_dirs[f] = p
    rows = []
    for _, r in after.merge(meta, on="CaseNumber").iterrows():
        if r["File"] in case_dirs:
            rows.append(dict(source="IARC", id=f"IARC-{r['CaseNumber']:03d}",
                             path=os.path.join(case_dirs[r["File"]], r["File"]),
                             via=r["VIA"], histology=r["Histology findings"]))
    jh = pd.read_csv(jhpiego_labels_csv)
    for _, r in jh.iterrows():
        rows.append(dict(source="Jhpiego", id=f"JH-{r['card']:03d}",
                         path=os.path.join(root, "via", r["file"]),
                         via=r["via"], histology=None))
    df = pd.DataFrame(rows)
    df = df[df["via"].isin(CLASSES)].reset_index(drop=True)
    df["label"] = df["via"].map(CLASSES.index)
    df["refer"] = (df["label"] > 0).astype(int)
    return df


def open_image(row):
    img = Image.open(row["path"])
    return crop_flashcard(img) if row["source"] == "Jhpiego" else img.convert("RGB")


# Jhpiego labels, read from the answer cards (Question 1: suspicious for cancer; Question 3: VIA result)
JHPIEGO_LABELS = '''file,card,via
via_img_001.jpg,1,Positive
via_img_002.jpg,2,Positive
via_img_003.jpg,3,Negative
via_img_04.jpg,4,Negative
via_img_005.jpg,5,Suspicious of cancer
via_img_006.jpg,6,Negative
via_img_007.jpg,7,Suspicious of cancer
via_img_008.jpg,8,Positive
via_img_009.jpg,9,Negative
via_img_010.jpg,10,Positive
via_img_011.jpg,11,Positive
via_img_012.jpg,12,Negative
via_img_013.jpg,13,Positive
via_img_014.jpg,14,Negative
via_img_015.jpg,15,Negative
via_img_016.jpg,16,Negative
via_img_017.jpg,17,Positive
via_img_018.jpg,18,Positive
via_img_019.jpg,19,Negative
via_img_020.jpg,20,Positive
via_img_021.jpg,21,Positive
via_img_022.jpg,22,Negative
via_img_023.jpg,23,Positive
via_img_024.jpg,24,Positive
via_img_025.jpg,25,Suspicious of cancer
via_img_026.jpg,26,Negative
via_img_027.jpg,27,Negative
via_img_028.jpg,28,Negative
via_img_029.jpg,29,Positive
via_img_030.jpg,30,Negative
via_img_031.jpg,31,Negative
via_img_032.jpg,32,Positive
via_img_033.jpg,33,Negative
via_img_034.jpg,34,Negative
via_img_035.jpg,35,Positive
via_img_036.jpg,36,Positive
via_img_037.jpg,37,Negative
via_img_038.jpg,38,Positive
via_img_040.jpg,40,Negative
via_img_041.jpg,41,Negative
via_img_042.jpg,42,Positive
via_img_043.jpg,43,Positive
via_img_044.jpg,44,Negative
via_img_045.jpg,45,Positive
via_img_046.jpg,46,Positive
via_img_047.jpg,47,Negative
via_img_048.jpg,48,Negative
via_img_049.jpg,49,Positive
via_img_050.jpg,50,Negative
via_img_051.jpg,51,Negative
via_img_052.jpg,52,Positive
via_img_053.jpg,53,Negative
via_img_054.jpg,54,Negative
via_img_055.jpg,55,Suspicious of cancer
via_img_056.jpg,56,Negative
via_img_057.jpg,57,Negative
via_img_058.jpg,58,Negative
via_img_059.jpg,59,Positive
via_img_060.jpg,60,Negative
via_img_061.jpg,61,Positive
via_img_062.jpg,62,Negative
via_img_063.jpg,63,Positive
via_img_064.jpg,64,Positive
via_img_065.jpg,65,Suspicious of cancer
via_img_066.jpg,66,Negative
via_img_067.jpg,67,Negative
via_img_068.jpg,68,Positive
via_img_069.jpg,69,Positive
via_img_070.jpg,70,Positive
via_img_071.jpg,71,Negative
via_img_072.jpg,72,Positive
via_img_073.jpg,73,Positive
via_img_074.jpg,74,Negative
via_img_075.jpg,75,Negative
via_img_076.jpg,76,Negative
via_img_077.jpg,77,Positive
via_img_078.jpg,78,Positive
via_img_079.jpg,79,Negative
via_img_080.jpg,80,Positive
via_img_081.jpg,81,Positive
via_img_082.jpg,82,Positive
via_img_083.jpg,83,Negative
via_img_084.jpg,84,Negative
via_img_085.jpg,85,Positive
via_img_086.jpg,86,Negative
via_img_087.jpg,87,Positive
via_img_088.jpg,88,Negative
via_img_089.jpg,89,Suspicious of cancer
via_img_090.jpg,90,Positive
via_img_091.jpg,91,Positive
via_img_092.jpg,92,Positive
via_img_093.jpg,93,Positive
via_img_094.jpg,94,Negative
via_img_095.jpg,95,Negative
via_img_096.jpg,96,Positive
via_img_097.jpg,97,Negative
via_img_098.jpg,98,Positive
via_img_099.jpg,99,Negative
via_img_100.jpg,100,Positive
via_img_101.jpg,101,Positive
via_img_102.jpg,102,Positive
via_img_103.jpg,103,Negative
via_img_104.jpg,104,Negative
via_img_105.jpg,105,Negative
via_img_106.jpg,106,Negative
via_img_107.jpg,107,Positive
via_img_108.jpg,108,Negative
via_img_109.jpg,109,Negative
via_img_110.jpg,110,Positive
via_img_111.jpg,111,Negative
via_img_112.jpg,112,Positive
via_img_113.jpg,113,Positive
via_img_114.jpg,114,Negative
via_img_115.jpg,115,Negative
via_img_116.jpg,116,Positive'''
with open('jhpiego_labels.csv', 'w') as f:
    f.write(JHPIEGO_LABELS)

df = load_table(DATA_ROOT, 'jhpiego_labels.csv')
print(len(df), 'images')
display(pd.crosstab(df['source'], df['via'], margins=True))

In [ ]:
# Load every image once (crop Jhpiego photos, shrink to 400 px short side) so training doesn't re-read Drive
def shrink(img, short=400):
    w, h = img.size
    s = short / min(w, h)
    return img.resize((round(w * s), round(h * s)))

t0 = time.time()
images = [shrink(open_image(r)) for _, r in df.iterrows()]
labels = df['label'].to_numpy()
refer = df['refer'].to_numpy()
print(f'loaded {len(images)} images in {time.time() - t0:.0f}s')

fig, axes = plt.subplots(3, 6, figsize=(15, 7))
for row, cls in enumerate(CLASSES):
    idx = df.index[df['via'] == cls].to_numpy()
    pick = np.random.default_rng(0).choice(idx, 6, replace=False)
    for ax, i in zip(axes[row], pick):
        ax.imshow(images[i]); ax.set_title(f"{df.source[i]}: {cls}", fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(), T.RandomRotation(15),
    T.ColorJitter(0.2, 0.2, 0.1, 0.02),  # small hue change: acetowhite colour is the signal
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(IMG), T.CenterCrop(IMG), T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = np.asarray(idx), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(images[j]), labels[j]

strata = (df['source'] + '_' + df['label'].astype(str)).to_numpy()
folds = np.zeros(len(df), dtype=int)
for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df, strata)):
    folds[te] = k
print(pd.crosstab(folds, df['via']))

## Metrics

In [ ]:
def sens_at_spec(y, s, spec):
    fpr, tpr, thr = roc_curve(y, s)
    ok = np.where(fpr <= 1 - spec + 1e-9)[0]
    j = ok[np.argmax(tpr[ok])]
    return tpr[j], thr[j]

def bootstrap_ci(y, s, fn, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        vals.append(fn(y[i], s[i]))
    return np.percentile(vals, [2.5, 97.5])

def crossfit_operating_point(y, s, spec):
    # Threshold for each fold is picked on the other folds' predictions, so the
    # reported sensitivity/specificity are not tuned on the images they score.
    pred = np.zeros_like(y)
    for k in range(N_FOLDS):
        _, t = sens_at_spec(y[folds != k], s[folds != k], spec)
        pred[folds == k] = s[folds == k] >= t
    sens = (pred[y == 1] == 1).mean()
    spec_ = (pred[y == 0] == 0).mean()
    return sens, spec_

def summarize(name, probs, extra=None):
    s = probs[:, 1] + probs[:, 2]
    auc = roc_auc_score(refer, s)
    auc_ci = bootstrap_ci(refer, s, roc_auc_score)
    sens92, _ = sens_at_spec(refer, s, STRETCH_SPEC)
    sens92_ci = bootstrap_ci(refer, s, lambda y, v: sens_at_spec(y, v, STRETCH_SPEC)[0])
    sens87, _ = sens_at_spec(refer, s, MIN_SPEC)
    sens87_ci = bootstrap_ci(refer, s, lambda y, v: sens_at_spec(y, v, MIN_SPEC)[0])
    cf_sens, cf_spec = crossfit_operating_point(refer, s, MIN_SPEC)
    pred3 = probs.argmax(1)
    row = {
        'model': name,
        'AUC': f'{auc:.3f} ({auc_ci[0]:.2f}-{auc_ci[1]:.2f})',
        'sens @ 92% spec': f'{sens92:.2f} ({sens92_ci[0]:.2f}-{sens92_ci[1]:.2f})',
        'sens @ 86.7% spec': f'{sens87:.2f} ({sens87_ci[0]:.2f}-{sens87_ci[1]:.2f})',
        'held-out sens / spec': f'{cf_sens:.2f} / {cf_spec:.2f}',
        'AUC IARC': f"{roc_auc_score(refer[df.source == 'IARC'], s[df.source == 'IARC']):.3f}",
        'AUC Jhpiego': f"{roc_auc_score(refer[df.source == 'Jhpiego'], s[df.source == 'Jhpiego']):.3f}",
        '3-class acc': f'{accuracy_score(labels, pred3):.3f}',
        '3-class macro-F1': f"{f1_score(labels, pred3, average='macro'):.3f}",
        'min bar met': bool(sens87 >= MIN_SENS),
        'stretch met': bool(sens92 >= STRETCH_SENS and auc >= STRETCH_AUC),
    }
    row.update(extra or {})
    return row

results, oof = [], {}

## Experiment 1: frozen ImageNet features + logistic regression

Same idea as the proposal feasibility test: no fine-tuning, just a linear classifier on EfficientNet-B0 features.

In [ ]:
@torch.no_grad()
def extract_features(model, idx):
    model.eval()
    feats = []
    for x, _ in DataLoader(ImgDS(idx, eval_tf), batch_size=32, num_workers=NUM_WORKERS):
        x = x.to(device)
        feats.append(((model(x) + model(torch.flip(x, dims=[3]))) / 2).float().cpu().numpy())
    return np.concatenate(feats)

backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
backbone.classifier = nn.Identity()
backbone = backbone.to(device)
X = extract_features(backbone, np.arange(len(df)))

probs = np.zeros((len(df), 3))
for k in range(N_FOLDS):
    tr, te = folds != k, folds == k
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000, class_weight='balanced'))
    clf.fit(X[tr], labels[tr])
    probs[te] = clf.predict_proba(X[te])
oof['Frozen EfficientNet-B0 + LR'] = probs
results.append(summarize('Frozen EfficientNet-B0 + LR', probs, {'params (M)': 4.0, 'fp32 size (MB)': 16}))
display(pd.DataFrame(results).set_index('model').T)

## Experiments 2 and 3: fine-tuning EfficientNet-B0 and MobileNetV3-Large

In [ ]:
def make_model(name):
    if name == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    elif name == 'MobileNetV3-Large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 3)
    return m

def train_fold(name, tr_idx):
    model = make_model(name).to(device)
    head = [p for n, p in model.named_parameters() if n.startswith('classifier')]
    body = [p for n, p in model.named_parameters() if not n.startswith('classifier')]
    opt = torch.optim.AdamW([{'params': body, 'lr': LR * 0.3}, {'params': head, 'lr': LR}], weight_decay=1e-4)
    dl = DataLoader(ImgDS(tr_idx, train_tf), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR * 0.3, LR], total_steps=EPOCHS * len(dl), pct_start=0.15)
    counts = np.bincount(labels[tr_idx], minlength=3)
    weights = torch.tensor(len(tr_idx) / (3 * np.maximum(counts, 1)), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    for _ in range(EPOCHS):
        model.train()
        for x, y in dl:
            x, y = x.to(device), torch.as_tensor(y).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
    return model

@torch.no_grad()
def predict(model, idx):
    model.eval()
    out = []
    for x, _ in DataLoader(ImgDS(idx, eval_tf), batch_size=32, num_workers=NUM_WORKERS):
        x = x.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            p = model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)
        out.append((p / 2).cpu().numpy())
    return np.concatenate(out)

def run_cv(name):
    seed_all()
    probs = np.zeros((len(df), 3))
    t0 = time.time()
    for k in range(N_FOLDS):
        model = train_fold(name, np.where(folds != k)[0])
        probs[folds == k] = predict(model, np.where(folds == k)[0])
        print(f'{name} fold {k + 1}/{N_FOLDS} done ({time.time() - t0:.0f}s)')
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    return probs, {'params (M)': round(n_params, 1), 'fp32 size (MB)': round(n_params * 4, 1)}

In [ ]:
for name in ['EfficientNet-B0', 'MobileNetV3-Large']:
    probs, extra = run_cv(name)
    oof[name] = probs
    results.append(summarize(name, probs, extra))
res = pd.DataFrame(results).set_index('model')
display(res.T)
res.to_csv(f'{OUT}/results.csv')

In [ ]:
# Experiment 4: average the two fine-tuned models (no extra training)
name = 'Ensemble (EfficientNet-B0 + MobileNetV3)'
oof[name] = (oof['EfficientNet-B0'] + oof['MobileNetV3-Large']) / 2
results.append(summarize(name, oof[name], {'params (M)': 8.2, 'fp32 size (MB)': 32.8}))
res = pd.DataFrame(results).set_index('model')
display(res.T)
res.to_csv(f'{OUT}/results.csv')

## Results against the target

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for name, probs in oof.items():
    s = probs[:, 1] + probs[:, 2]
    fpr, tpr, _ = roc_curve(refer, s)
    ax[0].plot(fpr, tpr, label=f'{name} (AUC {roc_auc_score(refer, s):.2f})')
for (name, (se, sp)), m in zip(REFERENCE_POINTS.items(), ['X', 'P', 's']):
    ax[0].scatter(1 - sp, se, marker=m, s=90, zorder=5, label=name)
ax[0].add_patch(plt.Rectangle((0, MIN_SENS), 1 - MIN_SPEC, 1 - MIN_SENS, color='orange', alpha=0.12, label='Minimum bar'))
ax[0].add_patch(plt.Rectangle((0, STRETCH_SENS), 1 - STRETCH_SPEC, 1 - STRETCH_SENS, color='green', alpha=0.15, label='Stretch goal'))
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.7)
ax[0].set(xlabel='1 - specificity', ylabel='Sensitivity', title='Referral (VIA+ or suspicious) vs negative, 5-fold out-of-fold')
ax[0].legend(fontsize=7.5, loc='lower right')

best = max(oof, key=lambda n: roc_auc_score(refer, oof[n][:, 1] + oof[n][:, 2]))
cm = confusion_matrix(labels, oof[best].argmax(1))
ax[1].imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
short = ['Negative', 'Positive', 'Suspicious\nof cancer']
ax[1].set(xticks=range(3), yticks=range(3), xticklabels=short, yticklabels=short,
          xlabel='Predicted', ylabel='Expert VIA label', title=f'3-class confusion matrix: {best}')
plt.tight_layout()
plt.savefig(f'{OUT}/roc_vs_target.png', dpi=150)
plt.show()

## Like-for-like check on IARC cases with histology

For IARC cases with a biopsy result, compare the model's referral score and the expert VIA call against histology (CIN2+ vs less than CIN2).
These cases were mostly biopsied *because* they looked abnormal, so this subset is small and biased toward positives. Treat it as a sanity check, not a headline number.

In [ ]:
def cin2_plus(h):
    if not isinstance(h, str) or h == 'Not done':
        return np.nan
    h = h.lower()
    return float(any(k in h for k in ['hsil', 'carcinoma', 'adenocarcinoma']))

hist = df['histology'].map(cin2_plus)
mask = hist.notna().to_numpy()
y_h = hist[mask].astype(int).to_numpy()
expert = refer[mask]
print(f'{mask.sum()} IARC cases with histology: {y_h.sum()} CIN2+, {(1 - y_h).sum()} below CIN2')
print(f'Expert VIA vs histology: sensitivity {(expert[y_h == 1] == 1).mean():.2f}, specificity {(expert[y_h == 0] == 0).mean():.2f}')
for name, probs in oof.items():
    s = (probs[:, 1] + probs[:, 2])[mask]
    auc = roc_auc_score(y_h, s)
    lo, hi = bootstrap_ci(y_h, s, roc_auc_score)
    print(f'{name}: AUC vs histology {auc:.3f} ({lo:.2f}-{hi:.2f})')

In [ ]:
pd.DataFrame({'id': df['id'], 'source': df['source'], 'via': df['via'], 'fold': folds,
              **{f'{n} refer score': p[:, 1] + p[:, 2] for n, p in oof.items()}}).to_csv(f'{OUT}/oof_predictions.csv', index=False)
print('saved results.csv, roc_vs_target.png, oof_predictions.csv to', OUT)

## Findings (first run, Nov 2 2026, Colab T4)

| Model | AUC (95% CI) | Sensitivity at 86.7% specificity | Held-out sens / spec | 3-class accuracy |
|---|---|---|---|---|
| Frozen EfficientNet-B0 + LR | 0.773 (0.72–0.82) | 0.46 | 0.47 / 0.86 | 0.673 |
| Fine-tuned EfficientNet-B0 | 0.842 (0.80–0.88) | 0.61 | 0.61 / 0.88 | 0.720 |
| **Fine-tuned MobileNetV3-Large** | **0.856 (0.81–0.90)** | **0.64 (0.49–0.80)** | **0.62 / 0.88** | **0.730** |
| Average of the two | 0.862 (0.82–0.90) | 0.61 | 0.59 / 0.86 | 0.733 |

- **Target status:** no model meets the minimum bar yet (sensitivity ≥ 73% at 87% specificity); the best is 64%. MobileNetV3 clears the AUC part of the stretch goal (0.856 ≥ 0.85) but not its sensitivity part.
- **Fine-tuning matters:** it adds about 0.07–0.08 AUC over the frozen-feature setup used in the proposal.
- **The phone-sized model is as good as the larger one:** MobileNetV3 (4.2M parameters, about 17 MB before quantization) matches or beats EfficientNet-B0, which supports the cheap-phone direction. Averaging the two adds almost nothing.
- **Where it fails:** "suspicious of cancer" is caught well (22 of 26), but 53 of 126 VIA-positive images are called negative. Missed positives are the main gap to the target.
- **Data source gap:** AUC is 0.89 on IARC photos but about 0.80 on the scanned Jhpiego flashcards.
- **Histology check:** on the 92 IARC cases with a biopsy, AUC against CIN2+ is 0.68–0.72. Expert VIA on the same cases scores sensitivity 1.00 / specificity 0.11, which shows how biased this subset is (biopsies were mostly taken from VIA-positive women).
- **Compared with the prior team:** 0.73 three-class accuracy vs their ~0.80, on different splits and with cross-validation, so not directly comparable.

**Next steps to close the gap:** add the IARC colposcopy bank, pretrain on Intel & MobileODT, crop to the cervix before classifying, try higher input resolution, and test the segmentation route with AnnoCerv masks.